<a href="https://colab.research.google.com/github/mimimonna/m-moire/blob/main/notebook_memoire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌸 Assistant intelligent pour la santé menstruelle
## Approche hybride : Machine Learning + NLP

**Auteur :** Monna DABO  
**Projet :** Mémoire de Master

## Contexte du projet

Ce projet vise à développer un assistant intelligent dédié à la santé menstruelle.

Il combine deux approches :
- le Machine Learning pour analyser des données tabulaires
- le NLP pour répondre aux questions en langage naturel

L’objectif est de fournir des réponses fiables et accessibles aux utilisatrices.
Ce notebook présente la démarche de création d'un assistant intelligent capable de traiter des données de santé féminine (cycles) et de répondre à des questions médicales via du NLP.

## Import des bibliothèques

Ce code permet d’importer les bibliothèques principales utilisées pour manipuler les données.
Pandas est utilisé pour les DataFrames, tandis que NumPy permet de gérer les opérations numériques.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

#NLP / Deep Learning
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer, util

## Chargement des données

Ce code permet de charger le dataset et d’afficher les premières lignes.
Cela permet de vérifier la structure des données et les variables disponibles.

In [ ]:
import pandas as pd

#Datasets numériques
period_log = pd.read_csv("Period_Log.csv")
user_profile = pd.read_csv("User_Profile.csv")

#Datasets textuels
train_text = pd.read_csv("Training Data.csv")
test_text = pd.read_csv("Testing Data.csv")

#Vérification rapide
print(period_log.head())
print(user_profile.head())
print(train_text.head())
print(test_text.head())

In [ ]:
#Dataset numérique
print("Cycle Dataset info:")
period_log.info()
print("\nDimensions:", period_log.shape)

#Dataset textuel
print("Health Dataset info:")
train_text.info()
print("\nDimensions:", train_text.shape)

In [ ]:
#Dataset numérique
period_log.head()

#Dataset textuel
train_text.head()

## Pré-traitement des données

Cette étape consiste à supprimer les valeurs manquantes afin d’éviter des erreurs lors de l’entraînement des modèles.

In [ ]:
from sklearn.preprocessing import LabelEncoder

#Prétraitement Cycle Dataset (ML)
#Encodage colonnes catégorielles
categorical_cols = ['cycle_phase', 'flow_level', 'pms_symptoms', 'ovulation_result']

period_log_processed = period_log.copy()

for col in categorical_cols:
    le = LabelEncoder()
    period_log_processed[col] = le.fit_transform(period_log_processed[col])

#Remplir les valeurs manquantes
period_log_processed['prev_cycle_length'].fillna(period_log_processed['prev_cycle_length'].mean(), inplace=True)

#Sélection features / target
features = ['prev_cycle_length', 'cycle_phase', 'flow_level', 'pain_level',
            'pms_symptoms', 'mood_score', 'stress_score_cycle', 'sleep_hours_cycle',
            'energy_level', 'concentration_score', 'work_hours_lost',
            'estrogen_pgml', 'progesterone_ngml', 'overall_health_score']
target = 'cycle_length_days'

X = period_log_processed[features]
y = period_log_processed[target]


Ces commandes affichent des informations détaillées sur chaque DataFrame (`period_log` et `train_text`), y compris le nombre d'entrées, le nombre de colonnes, les types de données et le nombre de valeurs non nulles. Cela permet d'avoir un aperçu rapide de la qualité et de la structure des données.

## Analyse exploratoire des données

Ce graphique permet de visualiser la distribution de la durée du cycle menstruel.
Il permet de vérifier si les données sont cohérentes.

Les données sont normalisées en utilisant `StandardScaler` pour centrer et réduire les caractéristiques ce qui est crucial pour les algorithmes basés sur la distance. Ensuite, les données sont divisées en ensembles d'entraînement et de test pour évaluer la performance du modèle de Machine Learning de manière impartiale.

In [ ]:
#Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Split train / test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


Le modèle Random Forest est un ensemble d’arbres de décision.
Il permet de capturer des relations complexes entre les variables.

Ce code permet d’obtenir les prédictions du modèle Random Forest.

Cette cellule entraîne un modèle `RandomForestClassifier` sur les données prétraitées et évalue ses performances à l'aide du rapport de classification, qui inclut la précision, le rappel, le score F1 et le support pour chaque classe.

In [ ]:
#Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("ML - Classification Report:")
print(classification_report(y_test, y_pred))

Ce bloc de code effectue un prétraitement sur les données textuelles en définissant une fonction `clean_text`. Cette fonction convertit le texte en minuscules, supprime les espaces multiples et ne conserve que les caractères alphanumériques et la ponctuation de base. Les colonnes `instruction` et `output` sont ensuite nettoyées en utilisant cette fonction.

In [ ]:
#Prétraitement Health Dataset (NLP)
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9?.! ]', '', text)
    return text

train_text['instruction_clean'] = train_text['instruction (string)'].apply(clean_text)
train_text['output_clean'] = train_text['output (string)'].apply(clean_text)

### Préparation des embeddings avec Sentence-BERT

Cette section utilise `SentenceTransformer` pour convertir les questions et les réponses textuelles en vecteurs numériques (embeddings). Ces embeddings capturent la signification sémantique des textes permettant ainsi de trouver les questions et réponses les plus similaires en utilisant la similarité cosinus.

In [ ]:
#Deep Learning / Sentence-BERT
model = SentenceTransformer('all-MiniLM-L6-v2')

instruction_embeddings = model.encode(train_text['instruction_clean'], convert_to_tensor=True)
output_embeddings = model.encode(train_text['output_clean'], convert_to_tensor=True)


### Recherche de réponses par similarité sémantique

Ce code illustre comment, à partir d'une question utilisateur, on peut trouver la question la plus sémantiquement proche dans notre dataset. La similarité cosinus est utilisée pour mesurer cette proximité entre l'embedding de la question utilisateur et les embeddings de toutes les questions du dataset. La réponse correspondante est ensuite récupérée.

In [ ]:
#Exemple de recherche d’une réponse
user_question = "How can I reduce menstrual cramps?"
user_embedding = model.encode(user_question.lower(), convert_to_tensor=True)

#Similarité cosine
cos_scores = util.cos_sim(user_embedding, instruction_embeddings)
best_idx = cos_scores.argmax().item() # Convert tensor to Python scalar

print("Question la plus proche :", train_text['instruction (string)'][best_idx])
print("Réponse correspondante :", train_text['output (string)'][best_idx])

In [ ]:
#Visualisations rapides
#Histogramme des longueurs de cycle
plt.figure(figsize=(8,5))
sns.histplot(period_log_processed['cycle_length_days'], bins=15, kde=True)
plt.title("Distribution des longueurs de cycle")
plt.xlabel("Cycle length (days)")
plt.show()

In [ ]:
#Boxplot de la longueur du cycle selon le flux
plt.figure(figsize=(8,5))
sns.boxplot(x='flow_level', y='cycle_length_days', data=period_log_processed)
plt.title("Longueur du cycle selon le flux")
plt.xlabel("Flow Level")
plt.ylabel("Cycle Length (days)")
plt.show()

In [ ]:
#Histogramme de la douleur (pain_level)
plt.figure(figsize=(8,5))
sns.histplot(period_log_processed['pain_level'], bins=10, kde=True)
plt.title("Distribution de la douleur (pain_level)")
plt.xlabel("Pain Level")
plt.show()

In [ ]:
#Heatmap des corrélations numériques
plt.figure(figsize=(12,8))

#Sélectionner uniquement les colonnes numériques pour la corrélation
numeric_cols = period_log_processed.select_dtypes(include=['number'])
sns.heatmap(numeric_cols.corr(), annot=True, fmt=".2f", cmap="coolwarm")

plt.title("Corrélations entre variables numériques")
plt.show()

In [ ]:
#Scatter plot : énergie vs concentration
plt.figure(figsize=(8,5))
sns.scatterplot(x='energy_level', y='concentration_score', hue='cycle_phase', data=period_log_processed)
plt.title("Énergie vs concentration selon phase du cycle")
plt.show()

In [ ]:
#Nombre de mots par question
train_text['num_words'] = train_text['instruction_clean'].apply(lambda x: len(x.split()))
plt.figure(figsize=(8,5))
sns.histplot(train_text['num_words'], bins=15, kde=True)
plt.title("Distribution du nombre de mots par question")
plt.xlabel("Nombre de mots")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=period_log_processed[['cycle_length_days','pain_level','stress_score_cycle']])
plt.title("Détection des outliers")
plt.show()

In [ ]:
train_text['answer_len'] = train_text['output_clean'].apply(lambda x: len(x.split()))

plt.figure(figsize=(8,5))
sns.histplot(train_text['answer_len'], bins=20, kde=True)
plt.title("Longueur des réponses")
plt.xlabel("Nombre de mots")
plt.show()

In [ ]:
#Flow level
plt.figure(figsize=(6,4))
sns.countplot(x='flow_level', data=period_log_processed)
plt.title("Répartition du flow level")
plt.show()

In [ ]:
#Cycle phase
plt.figure(figsize=(6,4))
sns.countplot(x='cycle_phase', data=period_log_processed)
plt.title("Répartition des phases du cycle")
plt.xticks(rotation=45)
plt.show()

L’analyse des importances de variables révèle que les niveaux hormonaux (progestérone et œstrogène), ainsi que la régularité du cycle, constituent les principaux déterminants de la prédiction de l’ovulation.
Ces résultats sont cohérents avec la littérature médicale sur la physiologie du cycle menstruel.

In [ ]:
#Ovulation result
plt.figure(figsize=(6,4))
sns.countplot(x='ovulation_result', data=period_log_processed)
plt.title("Résultat d'ovulation")
plt.show()

In [ ]:
period_log_processed['ovulation_result'].value_counts()

Ici, la variable cible (`y`) est définie comme la colonne `ovulation_result`. Les caractéristiques (`X`) sont créées en supprimant la variable cible, ainsi que les colonnes `user_id` et `start_date` qui ne sont pas pertinentes pour l'entraînement du modèle.

In [ ]:
#Target
y = period_log_processed['ovulation_result']

#Features (on enlève ce qu'il ne faut pas)
X = period_log_processed.drop(columns=[
    'ovulation_result',
    'user_id',
    'start_date'
])

Cette ligne utilise `pd.get_dummies` pour effectuer un encodage one-hot des colonnes catégorielles restantes dans le DataFrame `X`. L'argument `drop_first=True` est utilisé pour éviter la multicolinéarité.

In [ ]:
X = pd.get_dummies(X, drop_first=True)

Les données (`X` et `y`) sont divisées en ensembles d'entraînement et de test avec un ratio de 80/20. `random_state=42` assure la reproductibilité, et `stratify=y` maintient la même proportion de classes cibles dans les ensembles d'entraînement et de test, ce qui est important pour les problèmes de classification.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Cette cellule entraîne un modèle de régression logistique (`LogisticRegression`) sur les données d'entraînement. Après l'entraînement, il effectue des prédictions sur l'ensemble de test et évalue la performance du modèle en affichant l'exactitude (accuracy), le rapport de classification et la matrice de confusion.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

#Modèle
model_lr = LogisticRegression(max_iter=1000)

#Entraînement
model_lr.fit(X_train, y_train)

#Prédictions
y_pred = model_lr.predict(X_test)

#Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

Cette cellule entraîne un modèle `RandomForestClassifier` avec 200 estimateurs sur les données d'entraînement. Les performances du modèle sont ensuite évaluées et affichées en utilisant l'exactitude, le rapport de classification et la matrice de confusion.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#Modèle
model_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

#Entraînement
model_rf.fit(X_train, y_train)

#Prédictions
y_pred_rf = model_rf.predict(X_test)

#Évaluation
print("Accuracy RF:", accuracy_score(y_test, y_pred_rf))
print("\nClassification report RF:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion matrix RF:\n", confusion_matrix(y_test, y_pred_rf))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues")
plt.title("Matrice de confusion - Random Forest")
plt.xlabel("Prédictions")
plt.ylabel("Valeurs réelles")
plt.show()

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Purples")
plt.title("Matrice de confusion - Régression logistique")
plt.xlabel("Prédictions")
plt.ylabel("Valeurs réelles")
plt.show()

Ce code calcule et visualise l'importance des caractéristiques (`feature_importances_`) pour le modèle Random Forest entraîné. Un graphique à barres est utilisé pour afficher les 10 caractéristiques les plus importantes ce qui aide à comprendre quelles variables contribuent le plus aux prédictions du modèle.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importances = model_rf.feature_importances_
feat_names = X.columns

feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(8,5))
feat_imp.head(10).plot(kind='bar')
plt.title("Top 10 features importantes")
plt.show()

Cette cellule utilise la bibliothèque `joblib` pour sauvegarder le modèle Random Forest entraîné (`model_rf`) dans un fichier au format `.pkl`. Cela permet de réutiliser le modèle sans avoir à le réentraîner.

In [ ]:
import joblib

joblib.dump(model_rf, "rf_ovulation_model.pkl")

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
#Modèle XGBoost
model_xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

#Train
model_xgb.fit(X_train, y_train)

#Predict
y_pred_xgb = model_xgb.predict(X_test)

#Evaluation
print("Accuracy XGB:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification report XGB:\n", classification_report(y_test, y_pred_xgb))
print("\nConfusion matrix XGB:\n", confusion_matrix(y_test, y_pred_xgb))

In [ ]:
!pip -q install transformers datasets evaluate accelerate sentencepiece

In [ ]:
#On garde les colonnes utiles
df = train_text[['instruction (string)', 'output (string)']].rename(
    columns={
        'instruction (string)': 'instruction',
        'output (string)': 'output'
    }
).dropna()

df.head()

In [ ]:
df.head(30)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42
)

In [ ]:
len(train_df), len(val_df)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

#Conversion en Dataset HuggingFace
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_len = 128
max_target_len = 192

def preprocess(batch):
    inputs = [
        "Answer the question about menstrual health: " + q
        for q in batch["instruction"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_len,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["output"],
        max_length=max_target_len,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

train_tok

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate
import numpy as np

#Install rouge_score dependency
!pip install rouge_score

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    preds[np.isinf(preds)] = tokenizer.pad_token_id 
    preds[np.isnan(preds)] = tokenizer.pad_token_id 

    preds = np.clip(preds, 0, tokenizer.vocab_size - 1).astype(np.int32)

    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(predictions=pred_text, references=label_text)

    return {
        "rouge1": scores["rouge1"],
        "rougeL": scores["rougeL"],
    }

training_args = Seq2SeqTrainingArguments(
    output_dir="flan_t5_menstrual",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,   #volontairement pour éviter overfitting
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=150,
    load_best_model_at_end=True
)

In [ ]:
def compute_metrics(eval_pred):

    preds, labels = eval_pred

    # gérer les tuples
    if isinstance(preds, tuple):
        preds = preds[0]

    #remplacer valeurs invalides
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=pred_text,
        references=label_text
    )

    return {
        "rouge1": scores["rouge1"],
        "rougeL": scores["rougeL"],
    }

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
from sentence_transformers import SentenceTransformer, util

#modèle embeddings
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

#embeddings des questions
questions = df["instruction"].tolist()
answers = df["output"].tolist()

question_embeddings = embed_model.encode(questions, convert_to_tensor=True)

In [ ]:
def retrieve_best_answer(user_question):
    user_emb = embed_model.encode(user_question, convert_to_tensor=True)
    scores = util.cos_sim(user_emb, question_embeddings)[0]

    best_idx = torch.argmax(scores).item()

    return answers[best_idx]

In [ ]:
def reformulate_answer(answer_text):

    prompt = f"""
    Rewrite the following medical answer clearly and completely.
    Keep all information.

    Original answer:
    {answer_text}

    Full rewritten answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        min_length=40,        
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        early_stopping=False 
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### Assistant Menstruel Intelligent (fonction principale)

La fonction `smart_menstrual_assistant` est le cœur de notre assistant. Elle combine la recherche sémantique avec la reformulation pour fournir la meilleure réponse possible. Elle inclut également un seuil de confiance (`threshold`) pour indiquer quand l'assistant n'est pas sûr de sa réponse et propose des alternatives si disponibles.

In [ ]:
def smart_menstrual_assistant(user_question, top_k=3, threshold=0.6):

    user_emb = embed_model.encode(user_question, convert_to_tensor=True)
    scores = util.cos_sim(user_emb, question_embeddings)[0]

    top_results = torch.topk(scores, k=top_k)

    best_score = top_results.values[0].item()

    if best_score < threshold:
        return {
            "final_answer": "I'm not confident enough to answer this question.",
            "alternative_answers": []
        }

    best_answers = [answers[idx.item()] for idx in top_results.indices]

    return {
        "final_answer": best_answers[0],
        "alternative_answers": best_answers[1:]
    }

### Test de l'assistant intelligent

Voici un exemple d'utilisation de la fonction `smart_menstrual_assistant` avec une question d'utilisateur pour démontrer son fonctionnement et le type de sortie attendue.

In [ ]:
result = smart_menstrual_assistant("I feel strong pain during my period, what can help me?")

print(result)